In [1]:
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [4]:
# 1. Load data
df = pd.read_csv("../data/imdb_preprocessed.csv")
X = df["processed_review"]
y = df["sentiment"].map({"negative": 0, "positive": 1})

In [5]:
# 2. Define a fresh Vectorizer instance (unfitted!)
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

In [6]:
# 3. Wrap models in Pipelines to automatically handle fitting inside each fold
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", tfidf),
        ("clf", LogisticRegression(random_state=42, max_iter=1000))
    ]),
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", tfidf),
        ("clf", MultinomialNB(alpha=0.1))
    ]),
    "Linear SVC": Pipeline([
        ("tfidf", tfidf),
        ("clf", LinearSVC(C=0.1,max_iter=2000,random_state=42))
    ])
}

In [7]:
# 4. Evaluate using Cross-Validation (Safe, zero data leakage!)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [8]:
results = []
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-2)
    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1 Score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "Accuracy Std": scores["test_accuracy"].std()
    })

In [9]:
results_df = pd.DataFrame(results).round(4)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Accuracy Std
0,Linear SVC,0.8928,0.8839,0.9053,0.8945,0.9591,0.0036
1,Logistic Regression,0.8923,0.8840,0.9040,0.8939,0.9587,0.0033
2,Multinomial Naive Bayes,0.8590,0.8485,0.8753,0.8617,0.9343,0.0032
